# Method: testing a claim without assuming a distribution

*Question → Intuition → Math → Code → Assumptions → How it breaks*

## 1. Question

"Henry was better than Suárez." It is a claim about two careers, and both left
evidence. Can the evidence settle it?

Not "is Henry's number bigger" — it is. The question is whether a gap that size
is more than two equally good players would produce by chance.

## 2. Intuition

Suppose the claim is false and the two are equally good. Then which career a
given season belongs to is an accident: any of those seasons could have been
either player's.

So **shuffle them**. Pool every season from both careers, deal them back out at
the original career lengths, and measure the gap. Do it ten thousand times. If
the real gap sits comfortably inside the pile of shuffled gaps, chance explains
it. If almost no shuffle reaches it, chance does not.

No bell curve appears anywhere in that paragraph. That is the point.

## 3. Math

Observed statistic, for careers $A$ (length $n_A$) and $B$:

$$t_{\text{obs}} = \bar{x}_A - \bar{x}_B$$

Under the null hypothesis of exchangeability, permute the pooled
$n_A + n_B$ values, split at $n_A$, and recompute $t$. Repeating $N$ times:

$$p = \frac{1 + \#\{t_b \ge t_{\text{obs}}\}}{N + 1}$$

**Why the $+1$ on both sides.** Without it, a claim no shuffle beats reports
$p = 0$ — asserting impossibility on the strength of ten thousand tries. The
correction makes the floor $1/(N+1)$, which is the strongest statement the
evidence can actually support.

**Why one-sided.** The claim is directional. Swapping the arguments tests the
opposite claim; two large p-values mean the pair is genuinely inseparable.

**Why not a t-test.** A t-test assumes the sampling distribution of the
difference is normal. A career is five to nineteen numbers, and the underlying
score distribution is visibly skewed. The permutation test assumes only that,
under the null, the labels are arbitrary — a far weaker claim, and one that is
actually plausible here.

In [ ]:
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")

In [ ]:
from gambeta import doubt


def career(name):
    return seasons[seasons["player"] == name]["season_score"].to_numpy()


a, b = "Thierry Henry", "Luis Suárez"
diff, p = doubt.permutation_test(career(a), career(b), n=10_000)
print(f"{a}: {len(career(a))} seasons, mean {career(a).mean():.3f}")
print(f"{b}: {len(career(b))} seasons, mean {career(b).mean():.3f}")
print(f"\nobserved gap {diff:+.3f}, p = {p:.4f}")

The null distribution, drawn. The observed gap is the vertical line;
the p-value is the share of the histogram to its right.

In [ ]:
left, right = career(a), career(b)
pooled = np.concatenate([left, right])
rng = np.random.default_rng(20260810)
shuffles = rng.permuted(np.tile(pooled, (10_000, 1)), axis=1)
gaps = shuffles[:, : left.size].mean(axis=1) - shuffles[:, left.size :].mean(axis=1)

fig, ax = plt.subplots()
ax.hist(gaps, bins=60, color="#B0B0B0", edgecolor="none", label="gaps chance produces")
ax.axvline(diff, color="#D62728", linewidth=2, label=f"observed gap {diff:+.3f}")
ax.set_xlabel(f"mean({a}) - mean({b}) after shuffling")
ax.set_ylabel("count")
ax.set_title("What the gap looks like when the players are equal")
ax.legend()
plt.show()

## Every pair in the top ten

One test is an anecdote. Run the whole table and the picture is different from
the one the ranking's ordering implies.

In [ ]:
top = ranking[ranking["qualified"]].head(8)["player"].tolist()
rows = []
for i, x in enumerate(top):
    for y in top[i + 1 :]:
        d, pv = doubt.permutation_test(career(x), career(y), n=5_000)
        rows.append({"claim": f"{x} > {y}", "gap": round(d, 3), "p": round(pv, 4)})

table = pd.DataFrame(rows).sort_values("p")
print(f"{(table['p'] < 0.05).sum()} of {len(table)} pairwise claims reach p < 0.05\n")
table.head(12)

## The multiple-comparisons trap

That table just ran 28 tests. At a 5% threshold, roughly **one in twenty tests
comes out significant when nothing is going on** — so a handful of "significant"
results in a table this size is exactly what pure noise looks like.

Demonstrate it on data with no signal at all.

In [ ]:
rng = np.random.default_rng(7)
fake = [rng.normal(0, 1, 12) for _ in range(8)]
false_positives = sum(
    1
    for i in range(len(fake))
    for j in range(i + 1, len(fake))
    if doubt.permutation_test(fake[i], fake[j], n=2_000)[1] < 0.05
)
n_tests = len(fake) * (len(fake) - 1) // 2
print(f"{n_tests} tests on players who are identical by construction")
print(f"{false_positives} came out 'significant' at p < 0.05")
print(f"\nBonferroni threshold for {n_tests} tests: {0.05 / n_tests:.4f}")

## 5. Assumptions

1. **Exchangeability under the null.** If the players were equal, any season
   could have belonged to either. Careers overlapping different eras strain this:
   a 2003 season and a 2023 season are not freely swappable, which is exactly why
   the scores are era-normalised first.
2. **Seasons are independent within a career.** Same caveat as the bootstrap.
3. **The statistic captures the claim.** A difference in means says nothing about
   peak, and "better" might mean peak.

## 6. How it breaks

**A p-value is not an effect size, and with enough data it stops being
interesting.** A trivial difference becomes "significant" once the sample is
large enough.

In [ ]:
rng = np.random.default_rng(3)
TRUE_GAP = 0.10  # a tenth of a standard deviation: nobody would see it on a pitch

for n in (50, 250, 1_000, 5_000):
    x, y = rng.normal(TRUE_GAP, 1, n), rng.normal(0, 1, n)
    _, pv = doubt.permutation_test(x, y, n=2_000)
    flag = "<- significant" if pv < 0.05 else ""
    print(f"n = {n:>5}: true gap {TRUE_GAP}, p = {pv:.4f}  {flag}")

print("\nThe difference never changed. Only the evidence for it did.")

**And a large p-value is not proof of equality.** "We cannot separate
these two" and "these two are the same" are different statements. The first is
about the evidence; the second is about the world. This project only ever makes
the first.